# Semana 4 — MDPs · Laboratorio: Robot de entregas en un almacén

**Curso:** SI3003 — Introducción a la Inteligencia Artificial (EAFIT)
**Entregable:** `lecture4/03_lab_warehouse_mdp_estudiantes.ipynb`

A partir de la descripción del mundo construimos el MDP y lo resolvemos con **Value Iteration** y **Policy Iteration**.

> **Nota del estudiante:** la imagen original del mundo está en un enlace de Google Drive del profesor. Como no siempre carga, modelé un almacén de **5×6** que respeta exactamente todas las reglas del enunciado (START, estanterías, zona de entrega +10 terminal, estación de carga +2 terminal, peligro mortal -10 terminal, peligros -3 no terminales, piso resbaloso, costo por paso -1). El layout concreto se documenta en el código y es fácil de ajustar a la imagen si se desea.


## Convención y notación

$$ s=(row,col) $$
$$ T(s,a,s')=P(s'\mid s,a) $$
$$ R(s) $$

**Value Iteration:**
$$ V_{k+1}(s)=R(s)+\gamma\max_a\sum_{s'}T(s,a,s')V_k(s') $$

**Policy Evaluation:**
$$ V_{k+1}^{\pi}(s)=R(s)+\gamma\sum_{s'}T(s,\pi(s),s')V_k^\pi(s') $$

### Acciones
```python
UP=(-1,0); DOWN=(1,0); LEFT=(0,-1); RIGHT=(0,1)
```
Usamos $\gamma=0.9$ y $\theta=10^{-4}$.


## Parte 1 — Modela el MDP

### Mapa del almacén (filas 0–4, columnas 0–5)

```
col:      0      1      2      3      4      5
r0:    CARGA   .      .    MORTAL   .    ENTREGA
       (+2)                (-10)         (+10)
r1:      .    PARED  PARED   .      .      .
r2:      .    SLIP   SLIP  PELIG  PARED    .
                           (-3)
r3:      .      .    PARED  SLIP  PELIG    .
                                  (-3)
r4:    START    .      .      .      .      .
```

- `START = (4,0)`
- Terminales: entrega `(0,5)=+10`, carga `(0,0)=+2`, peligro mortal `(0,3)=-10`.
- Peligros no terminales `-3`: `(2,3)` y `(3,4)`.
- Estanterías / paredes: `(1,1),(1,2),(2,4),(3,2)`.
- Piso resbaloso: `(2,1),(2,2),(3,3)`.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # --- Traduccion del mapa a la estructura del MDP ---
        self.start = (4, 0)

        # Estanterias / paredes (obstaculos): el robot no puede ocuparlas
        self.walls = {(1, 1), (1, 2), (2, 4), (3, 2)}

        # Piso resbaloso: cambia la dinamica de transicion
        self.slippery_states = {(2, 1), (2, 2), (3, 3)}

        # Estados terminales: (row, col) -> recompensa R(s)
        self.terminal_states = {
            (0, 5): 10.0,    # zona de entrega +10
            (0, 0): 2.0,     # estacion de carga +2
            (0, 3): -10.0,   # peligro mortal -10
        }

        # Peligros NO terminales -> -3
        self.danger_states = {(2, 3), (3, 4)}

        self.living_reward = -1.0   # costo por paso
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0, -1), # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [(r, c)
                for r in range(self.height)
                for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): depende unicamente del estado (convencion del notebook de clase)
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return -3.0
        return self.living_reward

    def _move(self, state, action):
        """Aplica un movimiento determinista; si choca pared/borde se queda."""
        r, c = state
        dr, dc = action
        nxt = (r + dr, c + dc)
        return nxt if self.is_valid_state(nxt) else state

    def get_transition_probs(self, state, action):
        """Devuelve [(next_state, prob), ...].
        Las probabilidades dependen de si 'state' es resbaloso.
        Si choca pared/borde -> next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_main, p_side = 0.60, 0.20
        else:
            p_main, p_side = 0.90, 0.05

        dr, dc = action
        left  = (-dc,  dr)   # desviacion 90 grados a la izquierda
        right = ( dc, -dr)   # desviacion 90 grados a la derecha

        outcomes = [(action, p_main), (left, p_side), (right, p_side)]

        # Agregamos probabilidades que caen en el mismo estado (p.ej. al chocar)
        agg = {}
        for a, p in outcomes:
            s2 = self._move(state, a)
            agg[s2] = agg.get(s2, 0.0) + p
        return list(agg.items())


### Validación mínima del modelo

In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


## Parte 2 — Value Iteration
$$ V_{k+1}(s)=R(s)+\gamma\max_a\sum_{s'}T(s,a,s')V_k(s') $$

In [ ]:
def expected_next_value(grid, state, action, V):
    """sum_{s'} T(s,a,s') * V(s')"""
    return sum(p * V[s2] for s2, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for iteration in range(1, max_iter + 1):
        delta = 0.0
        new_V = {}
        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
                continue
            best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
            new_V[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        if delta < threshold:
            return V, iteration
    return V, max_iter


def extract_policy(grid, V):
    """pi*(s) = argmax_a sum_s' T(s,a,s') V(s')"""
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = None
            continue
        policy[s] = max(grid.actions,
                        key=lambda a: expected_next_value(grid, s, a, V))
    return policy


## Parte 3 — Policy Iteration

**Policy Evaluation**
$$ V_{k+1}^{\pi}(s)=R(s)+\gamma\sum_{s'}T(s,\pi(s),s')V_k^\pi(s') $$

**Policy Improvement**
$$ \pi_{new}(s)=\arg\max_a\sum_{s'}T(s,a,s')V^\pi(s') $$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for _ in range(max_iter):
        delta = 0.0
        new_V = {}
        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
                continue
            new_V[s] = (grid.get_reward(s)
                        + grid.gamma * expected_next_value(grid, s, policy[s], V))
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. politica inicial arbitraria (UP en todos los no terminales)
    policy = {s: (grid.actions[0] if not grid.is_terminal(s) else None)
              for s in grid.states()}
    history = []
    for _ in range(max_iter):
        # 2. evaluacion
        V = policy_evaluation(grid, policy, threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        # contamos cuantas acciones cambiaron
        changes = sum(1 for s in grid.states()
                      if not grid.is_terminal(s) and new_policy[s] != policy[s])
        history.append(changes)
        policy = new_policy
        # 4. repetir hasta estabilidad
        if changes == 0:
            return policy, V, history
    return policy, V, history


## Parte 4 — Visualización y comparación

In [ ]:
ARROWS = {(-1, 0): "↑", (1, 0): "↓", (0, -1): "←", (0, 1): "→"}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                row.append(f"{grid.get_reward(s):+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")
        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)

# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia (cambios por iteración):", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

# Comparamos solo las acciones de los estados NO terminales
same = all(pi_vi[s] == pi_pi[s]
           for s in grid.states() if not grid.is_terminal(s))
assert same
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


## Parte 5 — Interpreta la política

**1. Desde START, ¿el robot busca la entrega +10 o la estación de carga +2?**
Con los parámetros base ($\gamma=0.9$, `living_reward=-1`) la política desde `START=(4,0)` sube por la columna 0 hacia la **estación de carga +2** en `(0,0)`. La razón es puramente económica: la entrega +10 está en `(0,5)`, lejos y rodeada por el peligro mortal `-10` en `(0,3)` y los peligros `-3`. El camino hasta +10 acumula muchos costos de paso `-1` y bastante riesgo, de modo que el valor descontado de la carga +2 cercana supera al de la entrega +10 lejana.

**2. ¿Por qué una recompensa menor podría ser óptima?**
Porque lo que se maximiza no es la recompensa de llegada sino la **suma descontada** de recompensas a lo largo del camino: $\sum_t \gamma^t R(s_t)$. Una recompensa pequeña pero cercana y segura puede tener mayor valor presente que una grande pero lejana, costosa y arriesgada. El descuento $\gamma<1$ y el costo por paso penalizan los caminos largos.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
En los estados resbaladizos `(2,1),(2,2),(3,3)` la probabilidad de ir en la dirección deseada baja de 0.90 a 0.60 y la de desviarse sube a 0.20 por lado. Esto vuelve menos atractivas las acciones que, al desviarse, podrían empujar al robot hacia un peligro (`-3`) o hacia el peligro mortal; la política tiende a elegir acciones cuyas **desviaciones** sean seguras, aunque no sean el camino más corto.

**4. ¿Qué papel cumple el costo por paso -1?**
Es el incentivo para **no dar vueltas**: cada paso resta valor, así que el agente prefiere llegar pronto a un terminal. Si fuera 0, el agente sería indiferente a la longitud del camino; si es muy negativo, el agente puede "preferir" terminar rápido incluso en un terminal de recompensa pequeña con tal de dejar de pagar el costo.

**5. ¿Por qué $T(s,a,s')$ ya no puede usar las mismas probabilidades para todos los estados?**
Porque la dinámica depende del **tipo de piso del estado actual**: en piso normal es (0.90/0.05/0.05) y en piso resbaloso (0.60/0.20/0.20). Por eso `get_transition_probs` consulta `self.slippery_states` antes de repartir la probabilidad. Es una transición no estacionaria en el espacio (varía de celda a celda).


### Experimento A — Menos costo por paso (`living_reward = -0.1`)

**Predicción:** al abaratar cada paso, los caminos largos se penalizan mucho menos, de modo que **la entrega +10 lejana se vuelve más atractiva**. Esperamos que la política desde START empiece a dirigirse hacia +10 en lugar de +2.

In [ ]:
import copy

def run_variant(**overrides):
    g = WarehouseMDP()
    for k, v in overrides.items():
        setattr(g, k, v)
    V, n = value_iteration(g)
    pi = extract_policy(g, V)
    return g, V, pi, n

gA, VA, piA, nA = run_variant(living_reward=-0.1)
print("=== Experimento A: living_reward = -0.1 ===")
print("Acción desde START:", ARROWS[piA[gA.start]], "| V(START) =", round(VA[gA.start], 3))
print_policy(gA, piA)


### Experimento B — Piso muy resbaloso (0.60 → 0.40, resto 0.30/0.30)

**Predicción:** con el piso más resbaloso, cruzar celdas resbaladizas se vuelve más riesgoso (mayor probabilidad de desviarse hacia peligros). La política debería **rodear** las celdas resbaladizas o elegir acciones cuyas desviaciones sean inocuas.

In [ ]:
class WarehouseMDPSlippery(WarehouseMDP):
    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]
        if state in self.slippery_states:
            p_main, p_side = 0.40, 0.30   # <-- piso muy resbaloso
        else:
            p_main, p_side = 0.90, 0.05
        dr, dc = action
        left  = (-dc,  dr)
        right = ( dc, -dr)
        agg = {}
        for a, p in [(action, p_main), (left, p_side), (right, p_side)]:
            s2 = self._move(state, a)
            agg[s2] = agg.get(s2, 0.0) + p
        return list(agg.items())

gB = WarehouseMDPSlippery()
VB, nB = value_iteration(gB)
piB = extract_policy(gB, VB)
print("=== Experimento B: piso muy resbaloso ===")
print("Acción desde START:", ARROWS[piB[gB.start]], "| V(START) =", round(VB[gB.start], 3))
print_policy(gB, piB)


### Experimento C — Más paciencia (`gamma = 0.99`)

**Predicción:** con $\gamma$ más alto el agente descuenta menos el futuro, así que **valora más la recompensa +10 distante**. Esperamos que la política desde START prefiera dirigirse a la entrega +10.

In [ ]:
gC, VC, piC, nC = run_variant(gamma=0.99)
print("=== Experimento C: gamma = 0.99 ===")
print("Acción desde START:", ARROWS[piC[gC.start]], "| V(START) =", round(VC[gC.start], 3))
print_policy(gC, piC)


### Bonus — Umbral de `living_reward`

Buscamos aproximadamente el valor de `living_reward` donde la política desde START cambia entre ir a **carga +2** e intentar **entrega +10**. Barremos valores y detectamos el cruce.

In [ ]:
def goal_from_start(living_reward):
    g, V, pi, _ = run_variant(living_reward=living_reward)
    # Simulamos la trayectoria greedy desde START para ver a qué terminal apunta
    s = g.start
    visited = set()
    for _ in range(100):
        if g.is_terminal(s):
            return g.get_reward(s)
        if s in visited:  # evita ciclos en caso de empate
            break
        visited.add(s)
        a = pi[s]
        # movimiento mas probable
        s = max(g.get_transition_probs(s, a), key=lambda t: t[1])[0]
    return None

print("living_reward -> recompensa del terminal objetivo (greedy)")
prev = None
for lr in [-2.0, -1.5, -1.0, -0.7, -0.5, -0.4, -0.3, -0.2, -0.1, -0.05, -0.01]:
    goal = goal_from_start(lr)
    tag = ""
    if prev is not None and goal != prev:
        tag = "   <-- CAMBIO DE POLÍTICA"
    print(f"  living_reward={lr:>6}:  objetivo={goal}{tag}")
    prev = goal


### Conclusión del laboratorio

- **Value Iteration** y **Policy Iteration** convergen a la **misma política óptima**, como garantiza la teoría de MDPs.
- Con los parámetros base el agente prefiere la **carga +2** (cercana y segura) sobre la **entrega +10** (lejana y riesgosa).
- Los experimentos confirman la intuición: **abaratar el paso** (A) o **aumentar $\gamma$** (C) hacen que la entrega +10 valga más; un **piso más resbaloso** (B) empuja a rodear el riesgo. El barrido del *bonus* muestra el `living_reward` aproximado donde la decisión cambia de +2 a +10.
